# **Semantic Web & Linked Open Data with Python**
## บทที่ 03 — RDFS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch03/ch03_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://www.w3.org/TR/rdf-schema/
*   https://owl-rl.readthedocs.io/

📖 เอกสารประกอบบทนี้: [`03-rdfs/`](https://github.com/Rathachai/semantic-web/tree/main/03-rdfs)

---

### Setup

In [1]:
!pip install -q rdflib owlrl

In [2]:
import rdflib, owlrl
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS

EX = Namespace('http://example.org/university#')
print(rdflib.__version__, owlrl.__version__ if hasattr(owlrl,'__version__') else 'owlrl')

7.6.0 7.6.2


## To Create a Class Hierarchy

`rdfs:subClassOf` สร้างลำดับชั้นของ class ได้

In [3]:
schema_ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex:   <http://example.org/university#> .

ex:Person a rdfs:Class .
ex:Student a rdfs:Class ; rdfs:subClassOf ex:Person .
ex:GraduateStudent a rdfs:Class ; rdfs:subClassOf ex:Student .

ex:enrolledIn a rdf:Property ; rdfs:domain ex:Student ; rdfs:range ex:Course .
ex:Course a rdfs:Class .
"""
g = Graph().parse(data=schema_ttl, format="turtle")
print(f"{len(g)} triples")

9 triples


## เพิ่มข้อมูล instance (ABox)

In [4]:
abox_ttl = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex:  <http://example.org/university#> .

ex:alice a ex:GraduateStudent .
ex:alice ex:enrolledIn ex:cs501 .
"""
g.parse(data=abox_ttl, format="turtle")
print(f"{len(g)} triples")

11 triples


## Query ก่อนรัน reasoner

`alice` ถูกประกาศเป็น `GraduateStudent` เท่านั้น -- ยังไม่รู้ว่าเป็น `Person` ด้วย

In [5]:
spql = "SELECT ?type WHERE { ex:alice rdf:type ?type . }"
res = g.query(spql, initNs={"ex": EX, "rdf": RDF})
for row in res:
    print(row.type)

http://example.org/university#GraduateStudent


## RDFS Reasoning ด้วย owlrl

`DeductiveClosure` จะเติม triple ที่อนุมานได้ทั้งหมดลงในกราฟโดยตรง

In [6]:
asserted = set(g)

In [7]:
owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)

In [8]:
inferred = set(g) - asserted
print(f'triple ใหม่ที่อนุมานได้: {len(inferred)}')

triple ใหม่ที่อนุมานได้: 32


## Query หลังรัน reasoner

ตอนนี้ `alice` ควรปรากฏเป็น `Student` และ `Person` ด้วยเพราะ **subClassOf transitivity**

In [9]:
res = g.query(spql, initNs={"ex": EX, "rdf": RDF})
for row in res:
    print(row.type)

http://example.org/university#GraduateStudent
http://example.org/university#Student
http://www.w3.org/2000/01/rdf-schema#Resource
http://example.org/university#Person


## กรอง noise ที่ RDFS สร้างมาเยอะเกินไป

reasoner จะเติม `rdfs:Resource`, `rdfs:Class` และ axiomatic triple อื่น ๆ ที่ไม่มีประโยชน์เชิงข้อมูล

In [10]:
NOISE = {RDFS.Resource, RDFS.Literal, RDFS.Class, RDF.Property}
useful = [(s, p, o) for s, p, o in inferred
          if o not in NOISE and str(s).startswith(str(EX))]
print(f"inferred ทั้งหมด: {len(inferred)}  --  ที่มีประโยชน์: {len(useful)}")
for t in sorted(useful, key=str)[:10]:
    print(" ", t)

inferred ทั้งหมด: 32  --  ที่มีประโยชน์: 9
  (rdflib.term.URIRef('http://example.org/university#Course'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/university#Course'))
  (rdflib.term.URIRef('http://example.org/university#GraduateStudent'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/university#GraduateStudent'))
  (rdflib.term.URIRef('http://example.org/university#GraduateStudent'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/university#Person'))
  (rdflib.term.URIRef('http://example.org/university#Person'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/university#Person'))
  (rdflib.term.URIRef('http://example.org/university#Student'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef(

## domain/range สร้าง type ให้ ไม่ใช่ข้อจำกัด

ลองเพิ่มคนที่ยังไม่เคยประกาศเป็น `Student` แต่ใช้ property `enrolledIn` ดู

In [11]:
g.add((EX.bob, EX.enrolledIn, EX.cs501))
owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)

res = g.query("SELECT ?type WHERE { ex:bob rdf:type ?type . }", initNs={"ex": EX, "rdf": RDF})
for row in res:
    print(row.type)
print("\n👀 bob ไม่เคยถูกประกาศเป็น Student ตรง ๆ แต่ rdfs:domain ทำให้กลายเป็น Student โดยอัตโนมัติ")

http://example.org/university#Student
http://www.w3.org/2000/01/rdf-schema#Resource
http://example.org/university#Person

👀 bob ไม่เคยถูกประกาศเป็น Student ตรง ๆ แต่ rdfs:domain ทำให้กลายเป็น Student โดยอัตโนมัติ


## 🎯 สรุป

- `rdfs:subClassOf` **transitive** — ถ้า `A ⊑ B ⊑ C` แล้ว `A ⊑ C` ด้วย
- `rdfs:domain`/`rdfs:range` **สร้าง type ให้** ไม่ใช่ตรวจสอบข้อจำกัด
- `owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)` รันแบบ **in-place**
- ต้องกรอง noise (`rdfs:Resource` ฯลฯ) เองถ้าต้องการดูเฉพาะ triple ที่มีความหมาย

ลองเพิ่ม class ใหม่และ subclass 2-3 ชั้น แล้วดูว่า reasoner ไล่ type ให้ครบไหม!

---

## จบบทที่ 03 🎉

ไปต่อบทถัดไป: **บทที่ 04 — Logics**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*